##Dataset

In [ ]:
import pandas as pd

In [ ]:
BASE_URL = "https://raw.githubusercontent.com/Profession-AI/progetti-ml/refs/heads/main/Modello%20di%20classificazione%20di%20pezzi%20difettosi%20in%20produzione/"

In [ ]:
df = pd.read_csv(BASE_URL+"parts_production_data.csv")
df.head()

## Analisi esplorativa del dataset (EDA)

In [ ]:
df.shape #quante righe, colonne

In [ ]:
df.info() #tipi di dato
df.describe() #statistiche descrittive

In [ ]:
df.isna().sum() #quanti valori mancanti

In [ ]:
df.nunique() #quanti valori unici

### Bilanciamento della classe target

In [ ]:
#quanti pezzi conformi (0) e difettosi (1) in numero assoluto
df['defect_label'].value_counts()

In [ ]:
#quanti pezzi conformi (0) e difettosi (1) in percentuale
df['defect_label'].value_counts(normalize=True)*100

Confronto anche la distribuzione di `flatness_mm` separando i pezzi conformi da quelli difettosi, per vedere a colpo d'occhio se le due distribuzioni si sovrappongono.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

print(df['flatness_mm'].describe())

sns.boxplot(x='defect_label', y='flatness_mm', data=df)
plt.show()

In [ ]:
#divido il dataset in base alla label binaria "defect"

df_defect_zero = df[df['defect_label'] == 0]
df_defect_one = df[df['defect_label'] == 1]

### Confronto delle Statistiche Descrittive per tutte le Variabili numeriche

### Correlazione tra le variabili numeriche e `defect_label`

Uso la correlazione point-biserial (equivalente a Pearson quando una delle due variabili è binaria) come primo screening: ci dice se esiste una relazione *lineare* significativa tra ciascuna variabile e l'etichetta, prima di scendere nei dettagli.

In [ ]:
import scipy.stats as stats

#confronto le variabili numeriche con la label "defect"
print("Correlazione tra 'defect_label' e altre variabili numeriche:")

numerical_cols = [
    'measure_diam_mm',
    'measure_length_mm',
    'surface_roughness_Ra',
    'temp_process_C',
    'cycle_time_s',
    'visual_inspection_score',
    'flatness_mm',
    'vibration_level',
    'torque_Nm'
]

correlation_results = {}

for col in numerical_cols:
    coefficiente, p_value = stats.pointbiserialr(df['defect_label'], df[col])
    correlation_results[col] = {'coefficient': coefficiente, 'p_value': p_value}

#trasformo il dizionario in un DataFrame
correlation_df = pd.DataFrame.from_dict(correlation_results, orient='index')
correlation_df.index.name = 'Variable'

#aggiungo la colonna "abs coefficient" (valore assoluto del coefficiente)
correlation_df['abs_coefficient'] = abs(correlation_df['coefficient'])
correlation_df_sorted = correlation_df.sort_values(by='abs_coefficient', ascending=False)

display(correlation_df_sorted)

### Confronto le 3 variabili più influenti con il Dataset diviso in base a "Defect" 1 o 0

**Risultato:** solo `vibration_level` e `cycle_time_s` mostrano una correlazione lineare statisticamente significativa (p-value basso); `temp_process_C` è appena al limite. Tutte le altre variabili, incluse `flatness_mm` e `torque_Nm`, non mostrano relazione lineare significativa con il difetto.

Confrontiamo le tre variabili con un segnale lineare per vedere come si distribuiscono separatamente tra pezzi conformi e difettosi.

In [ ]:
print("Statistiche descrittive per 'vibration_level' (No Defect):")
display(df_defect_zero['vibration_level'].describe())

print("\nStatistiche descrittive per 'vibration_level' (Defect):")
display(df_defect_one['vibration_level'].describe())

In [ ]:
print("Statistiche descrittive per 'cycle_time_s' (No Defect):")
display(df_defect_zero['cycle_time_s'].describe())

print("\nStatistiche descrittive per 'cycle_time_s' (Defect):")
display(df_defect_one['cycle_time_s'].describe())

In [ ]:
print("Statistiche descrittive per 'temp_process_C' (No Defect):")
display(df_defect_zero['temp_process_C'].describe())

print("\nStatistiche descrittive per 'temp_process_C' (Defect):")
display(df_defect_one['temp_process_C'].describe())

### Calcolo dei limiti per `flatness_mm` e `torque_Nm`

### Outlier su `flatness_mm` e `torque_Nm`

`flatness_mm` e `torque_Nm` non hanno mostrato correlazione lineare significativa, ma essendo le variabili più tecniche è importante capire se una tolleranza meccanica (più alta o più bassa) produce un effetto "a soglia" (il pezzo è difettoso se il valore esce da un range, non in modo proporzionale) che è un dato che la correlazione lineare non può cogliere. Verifico quindi se i valori estremi (outlier) di queste due variabili sono associati a un tasso di difettosità più alto, con due metodi distinti: IQR e deviazione standard.

In [ ]:
print("Statistiche descrittive per 'flatness_mm':")
display(df['flatness_mm'].describe())

print("Statistiche descrittive per 'torque_Nm':")
display(df['torque_Nm'].describe())

**IQR** = differenza tra primo quartile (Q1) e terzo quartile (Q3)

Tutto ciò che cade fuori da questo range è un potenziale outlier

In [ ]:
print("--- Calcolo per 'flatness_mm' ---")
Q1_flatness = df['flatness_mm'].quantile(0.25)  #metodo di Pandas
Q3_flatness = df['flatness_mm'].quantile(0.75)
IQR_flatness = Q3_flatness - Q1_flatness
lower_bound_flatness = Q1_flatness - 1.5 * IQR_flatness
upper_bound_flatness = Q3_flatness + 1.5 * IQR_flatness

print(f"Q1 (25° percentile) per flatness_mm: {Q1_flatness:.2f}")
print(f"Q3 (75° percentile) per flatness_mm: {Q3_flatness:.2f}")
print(f"IQR per flatness_mm: {IQR_flatness:.2f}")
print(f"Limite inferiore per flatness_mm: {lower_bound_flatness:.2f}")
print(f"Limite superiore per flatness_mm: {upper_bound_flatness:.2f}")

print("\n--- Calcolo per 'torque_Nm' ---")
Q1_torque = df['torque_Nm'].quantile(0.25)
Q3_torque = df['torque_Nm'].quantile(0.75)
IQR_torque = Q3_torque - Q1_torque
lower_bound_torque = Q1_torque - 1.5 * IQR_torque
upper_bound_torque = Q3_torque + 1.5 * IQR_torque

print(f"Q1 (25° percentile) per torque_Nm: {Q1_torque:.2f}")
print(f"Q3 (75° percentile) per torque_Nm: {Q3_torque:.2f}")
print(f"IQR per torque_Nm: {IQR_torque:.2f}")
print(f"Limite inferiore per torque_Nm: {lower_bound_torque:.2f}")
print(f"Limite superiore per torque_Nm: {upper_bound_torque:.2f}")

### Identificazione e Analisi degli Outlier per `flatness_mm`

In [ ]:
df['is_flatness_outlier'] = ((df['flatness_mm'] < lower_bound_flatness) | (df['flatness_mm'] > upper_bound_flatness))  # "|" equivale a True OR False

print(f"Numero di righe 'outlier' per flatness_mm: {df['is_flatness_outlier'].sum()}")

defect_mean_flatness = df.groupby('is_flatness_outlier')['defect_label'].mean()  #metodo di Pandas per raggruppare valori identici
print("\nMedia di 'defect_label' per flatness_mm (False = inlier, True = outlier):")
display(defect_mean_flatness)

### Identificazione e Analisi degli Outlier per `torque_Nm`

In [ ]:
df['is_torque_outlier'] = ((df['torque_Nm'] < lower_bound_torque) | (df['torque_Nm'] > upper_bound_torque))

print(f"Numero di righe 'outlier' per torque_Nm: {df['is_torque_outlier'].sum()}")

defect_mean_torque = df.groupby('is_torque_outlier')['defect_label'].mean()
print("\nMedia di 'defect_label' per torque_Nm (False = inlier, True = outlier):")
display(defect_mean_torque)

### Calcolo dei Limiti degli Outlier utilizzando la Deviazione Standard

In [ ]:
print("--- Calcolo per 'flatness_mm' (deviazione standard) ---")
mean_flatness = df['flatness_mm'].mean()
std_flatness = df['flatness_mm'].std()
lower_bound_flatness_std = mean_flatness - 3 * std_flatness
upper_bound_flatness_std = mean_flatness + 3 * std_flatness

print(f"Media per flatness_mm: {mean_flatness:.2f}")
print(f"Deviazione standard per flatness_mm: {std_flatness:.2f}")
print(f"Limite inferiore per flatness_mm (std): {lower_bound_flatness_std:.2f}")
print(f"Limite superiore per flatness_mm (std): {upper_bound_flatness_std:.2f}")

print("\n--- Calcolo per 'torque_Nm' (deviazione standard) ---")
mean_torque = df['torque_Nm'].mean()
std_torque = df['torque_Nm'].std()
lower_bound_torque_std = mean_torque - 3 * std_torque
upper_bound_torque_std = mean_torque + 3 * std_torque

print(f"Media per torque_Nm: {mean_torque:.2f}")
print(f"Deviazione standard per torque_Nm: {std_torque:.2f}")
print(f"Limite inferiore per torque_Nm (std): {lower_bound_torque_std:.2f}")
print(f"Limite superiore per torque_Nm (std): {upper_bound_torque_std:.2f}")

### Re-identificazione e Analisi degli Outlier per `flatness_mm` (con Deviazione Standard)

In [ ]:
df['is_flatness_outlier_std'] = ((df['flatness_mm'] < lower_bound_flatness_std) | (df['flatness_mm'] > upper_bound_flatness_std))

print(f"Numero di righe 'outlier' per flatness_mm (deviazione standard): {df['is_flatness_outlier_std'].sum()}")

defect_mean_flatness_std = df.groupby('is_flatness_outlier_std')['defect_label'].mean()
print("\nMedia di 'defect_label' per flatness_mm (False = inlier, True = outlier - std):")
display(defect_mean_flatness_std)

### Re-identificazione e Analisi degli Outlier per `torque_Nm` (con Deviazione Standard)

In [ ]:
df['is_torque_outlier_std'] = ((df['torque_Nm'] < lower_bound_torque_std) | (df['torque_Nm'] > upper_bound_torque_std))

print(f"Numero di righe 'outlier' per torque_Nm (deviazione standard): {df['is_torque_outlier_std'].sum()}")

defect_mean_torque_std = df.groupby('is_torque_outlier_std')['defect_label'].mean()
print("\nMedia di 'defect_label' per torque_Nm (False = inlier, True = outlier - std):")
display(defect_mean_torque_std)

**Risultato:** con entrambi i metodi (IQR e deviazione standard ±3σ), zero outlier per entrambe le variabili. Verifichiamo che non sia un errore di calcolo, controllando il range reale dei dati rispetto ai limiti trovati.

In [ ]:
print(f"Valore minimo di torque: {df['torque_Nm'].min():.2f}")
print(f"Valore massimo di torque: {df['torque_Nm'].max():.2f}")
print(f"Valore minimo di flatness: {df['flatness_mm'].min():.2f}")
print(f"Valore massimo di flatness: {df['flatness_mm'].max():.2f}")

### Tasso di difettosità per variabili categoriche (`line_id`, `station_id`, `operator_id`)

Le variabili numeriche prese singolarmente hanno mostrato segnali debolissimi. Controlliamo se le variabili categoriche a bassa cardinalità (`line_id`, `station_id`, `operator_id`) mostrano una variazione più marcata nel tasso di difettosità, controllando anche la dimensione di ciascun gruppo per essere sicuri che le differenze non siano dovute a campioni troppo piccoli.

In [ ]:
df.groupby('line_id')['defect_label'].mean()

In [ ]:
df.groupby('line_id').size()

In [ ]:
df.groupby('station_id')['defect_label'].mean()

In [ ]:
df.groupby('station_id').size()

In [ ]:
df.groupby('operator_id')['defect_label'].mean()

In [ ]:
df.groupby('operator_id').size()

**Risultato:** `line_id` e `station_id` mostrano una variazione di circa 9-10 punti percentuali tra le categorie, `operator_id` una variazione più debole (~5 punti) — tutti con gruppi di dimensione comparabile, quindi attendibili.

`material_batch` invece va controllato diversamente per l'altissima cardinalità trovata in precedenza (quasi 3000 valori distinti): verifichiamo quanti pezzi contiene in media ciascun lotto.

In [ ]:
df.groupby('material_batch').size().describe()

**Risultato:** la stragrande maggioranza dei lotti contiene 1 solo pezzo (massimo 2) — `material_batch` si comporta come un identificativo quasi univoco, non come una vera categoria, e va escluso dalle feature.

### Correlazione tra le variabili numeriche (multicollinearità)

Controlliamo infine se le variabili numeriche sono correlate *tra loro* (non più con l'etichetta): se due variabili portassero la stessa informazione, una delle due sarebbe ridondante per il modello.

In [ ]:
df[numerical_cols].corr()

**Risultato:** tutte le correlazioni tra le variabili numeriche sono vicine a zero (massimo ~0.05) — nessuna ridondanza da eliminare.

## Preparazione delle variabili

### Rimozione delle colonne non utilizzabili come feature, ed estrazione di feature dal timestamp

Tolgo `part_id` (identificativo univoco, nessun potere predittivo) e `material_batch` (cardinalità troppo alta, vista sopra). Converto `production_timestamp` in un vero formato data/ora e ne estraggo due feature più utilizzabili dal modello: l'ora del giorno e il giorno della settimana.

In [ ]:
df_model = df.drop(columns=['part_id', 'material_batch'])

# Converto 'production_timestamp' in datetime e estraggo features
df_model['production_timestamp'] = pd.to_datetime(df_model['production_timestamp'])
df_model['hour'] = df_model['production_timestamp'].dt.hour
df_model['day_of_week'] = df_model['production_timestamp'].dt.dayofweek

In [ ]:
df_model.shape

In [ ]:
df_model.head()

Le quattro colonne booleane create durante l'analisi degli outlier sono risultate sempre `False` (zero outlier trovati) — verifichiamo che siano quindi costanti, e quindi inutili come feature, prima di toglierle.

In [ ]:
df_model[['is_flatness_outlier', 'is_torque_outlier', 'is_flatness_outlier_std', 'is_torque_outlier_std']].sum()

Verifichiamo se il tasso di difettosità varia per ora del giorno o giorno della settimana, con lo stesso approccio già usato per le altre variabili categoriche.

In [ ]:
print(f"Media di 'defect_label' per 'hour':\n{df_model.groupby('hour')['defect_label'].mean()}\n")
print(f"Size del gruppo per 'hour':\n{df_model.groupby('hour').size()}\n")
print(f"Media di 'defect_label' per 'day_of_week':\n{df_model.groupby('day_of_week')['defect_label'].mean()}\n")
print(f"Size del gruppo per 'day_of_week':\n{df_model.groupby('day_of_week').size()}\n")

**Risultato:** un pattern a "U" per ora del giorno (più alto a inizio/fine turno) e una crescita quasi progressiva dal lunedì al venerdì — entrambi segnali modesti ma plausibili. La colonna `production_timestamp` originale non serve più al modello (ha già fornito `hour` e `day_of_week`), e le colonne degli outlier sono costanti: le tolgo tutte.

In [ ]:
df_model = df_model.drop(columns=['production_timestamp',
                                  'is_flatness_outlier',
                                  'is_torque_outlier',
                                  'is_flatness_outlier_std',
                                  'is_torque_outlier_std'])
df_model.head()

In [ ]:
df_model.columns

### Encoding delle variabili categoriche

`line_id`, `station_id` e `operator_id` hanno poche categorie ciascuna (bassa cardinalità): le trasformiamo con il one-hot encoding.

In [ ]:
df_model = pd.get_dummies(df_model, columns=['line_id', 'station_id', 'operator_id'])
df_model.shape

Verifica rapida: `vibration_level` e `visual_inspection_score` sembravano a prima vista "piccole" come le colonne 0/1 create dal one-hot encoding — controllo che siano in realtà misure continue (min/max diversi da 0 e 1) prima di includerle tra le variabili da scalare.

In [ ]:
df_model['vibration_level'].describe()

In [ ]:
df_model['visual_inspection_score'].describe()

### Scaling

### Separazione di feature ed etichetta, e split train/test

In [ ]:
X = df_model.drop(columns=['defect_label'])
y = df_model['defect_label']

Uso `stratify=y` per mantenere la stessa proporzione 77.5%/22.5% di conformi/difettosi sia nel training che nel test set, evitando che una divisione puramente casuale la sbilanci per puro caso.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(X_train.shape, X_test.shape)
print(y_train.value_counts(normalize=True))
print(y_test.value_counts(normalize=True))

### Scaling delle variabili numeriche

Faccio lo scaling *dopo* lo split, calcolando media e deviazione standard solo sul training set (`fit_transform`) e applicando gli stessi parametri al test set (solo `transform`, senza un nuovo fit) — altrimenti il test set contribuirebbe a calcolare i parametri usati per trasformare anche se stesso (data leakage). Decision Tree e Random Forest non ne avrebbero bisogno (lavorano per soglie, insensibili alla scala), ma non succede niente ad applicarlo comunque; per Logistic Regression invece è necessario.

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train[numerical_cols] = scaler.fit_transform(X_train[numerical_cols])
X_test[numerical_cols] = scaler.transform(X_test[numerical_cols])

In [ ]:
X_train[numerical_cols].describe()

In [ ]:
X_test[numerical_cols].describe()

##Addestramento modelli

###1) Decision Tree

In [ ]:
from sklearn.tree import DecisionTreeClassifier

dt_model = DecisionTreeClassifier(random_state=42)
dt_model.fit(X_train, y_train)

In [ ]:
y_pred_dt = dt_model.predict(X_test)
y_pred_dt[:10]

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

print("Accuracy:", accuracy_score(y_test, y_pred_dt))
print("Precision:", precision_score(y_test, y_pred_dt))
print("Recall:", recall_score(y_test, y_pred_dt))
print("F1:", f1_score(y_test, y_pred_dt))

###2) Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train, y_train)

In [ ]:
y_pred_rf = rf_model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred_rf))
print("Precision:", precision_score(y_test, y_pred_rf))
print("Recall:", recall_score(y_test, y_pred_rf))
print("F1:", f1_score(y_test, y_pred_rf))

### 3) Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression

lr_model = LogisticRegression(random_state=42, max_iter=1000)
lr_model.fit(X_train, y_train)

y_pred_lr = lr_model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred_lr))
print("Precision:", precision_score(y_test, y_pred_lr))
print("Recall:", recall_score(y_test, y_pred_lr))
print("F1:", f1_score(y_test, y_pred_lr))

###Modelli "bilanciati"

In [ ]:
lr_model_balanced = LogisticRegression(random_state=42, max_iter=1000, class_weight="balanced")
lr_model_balanced.fit(X_train, y_train)

y_pred_lr_balanced = lr_model_balanced.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred_lr_balanced))
print("Precision:", precision_score(y_test, y_pred_lr_balanced))
print("Recall:", recall_score(y_test, y_pred_lr_balanced))
print("F1:", f1_score(y_test, y_pred_lr_balanced))

In [ ]:
rf_model_balanced = RandomForestClassifier(random_state=42, class_weight="balanced")
rf_model_balanced.fit(X_train, y_train)

y_pred_rf_balanced = rf_model_balanced.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred_rf_balanced))
print("Precision:", precision_score(y_test, y_pred_rf_balanced))
print("Recall:", recall_score(y_test, y_pred_rf_balanced))
print("F1:", f1_score(y_test, y_pred_rf_balanced))

In [ ]:
dt_model_balanced = DecisionTreeClassifier(random_state=42, class_weight="balanced")
dt_model_balanced.fit(X_train, y_train)

y_pred_dt_balanced = dt_model_balanced.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred_dt_balanced))
print("Precision:", precision_score(y_test, y_pred_dt_balanced))
print("Recall:", recall_score(y_test, y_pred_dt_balanced))
print("F1:", f1_score(y_test, y_pred_dt_balanced))

###ROC-AUC

In [ ]:
from sklearn.metrics import roc_auc_score

y_proba_lr = lr_model.predict_proba(X_test)[:,1]
y_proba_lr_balanced = lr_model_balanced.predict_proba(X_test)[:,1]
y_proba_dt = dt_model.predict_proba(X_test)[:, 1]
y_proba_dt_balanced = dt_model_balanced.predict_proba(X_test)[:,1]
y_proba_rf = rf_model.predict_proba(X_test)[:,1]
y_proba_rf_balanced = rf_model_balanced.predict_proba(X_test)[:,1]
print(f"ROC-AUC 'Logistic Regression': {roc_auc_score(y_test, y_proba_lr):.2f}")
print(f"ROC-AUC 'Logistic Regression balanced': {roc_auc_score(y_test, y_proba_lr_balanced):.2f}")
print(f"ROC-AUC 'Decision Tree': {roc_auc_score(y_test, y_proba_dt):.2f}")
print(f"ROC-AUC 'Decision Tree balanced': {roc_auc_score(y_test, y_proba_dt_balanced):.2f}")
print(f"ROC-AUC 'Random Forest': {roc_auc_score(y_test, y_proba_rf):.2f}")
print(f"ROC-AUC 'Random Forest balanced': {roc_auc_score(y_test, y_proba_rf_balanced):.2f}")


### Soglia di decisione personalizzata

Il Random Forest ha il ROC-AUC più alto (sa ordinare il rischio meglio degli altri) ma a soglia 0.5 ha un recall pessimo: la soglia di default è troppo alta per come distribuisce le sue probabilità. Proviamo ad abbassarla, dato che per AutomaParts un pezzo difettoso non individuato (falso negativo) è l'errore più costoso.

In [ ]:
soglia = 0.2
y_pred_rf_soglia = (y_proba_rf >= soglia).astype(int)

print("Accuracy:", accuracy_score(y_test, y_pred_rf_soglia))
print("Precision:", precision_score(y_test, y_pred_rf_soglia))
print("Recall:", recall_score(y_test, y_pred_rf_soglia))
print("F1:", f1_score(y_test, y_pred_rf_soglia))

**Risultato:** abbassando la soglia il recall sale molto (più pezzi difettosi individuati) a scapito della precision e dell'accuracy — un trade-off che si decide in base al costo operativo di un richiamo cliente rispetto a un controllo extra.

###Feature Importance

In [ ]:
importanze = pd.Series(rf_model.feature_importances_, index=X_train.columns)
importanze_ordinate = importanze.sort_values(ascending=False)
importanze_ordinate.head(15)

## Conclusioni e raccomandazioni

### Sintesi dei risultati e limiti
L'analisi ha rivelato un moderato squilibrio nella classificazione dei pezzi, con circa il **22%** segnalato come difettoso. Il `vibration_level` e il `cycle_time_s` mostrano una **relazione statisticamente significativa con il difetto, sebbene di intensità modesta**. Il `temp_process_C` aveva un segnale meno forte, da interpretare con cautela. Per `flatness_mm` e `torque_Nm` non ho trovato né valori anomali (outlier) né forti correlazioni lineari con la presenza di difetti, eppure il modello le ha considerate importanti.

Il **Random Forest** si è dimostrato il modello più promettente. La sua capacità di distinguere tra pezzi conformi e difettosi è stata **solo leggermente migliore del puro caso** (ROC-AUC di 0.58-0.59). Per AutomaParts, azienda di produzione di pezzi per autovetture, il rischio maggiore sarebbe di non identificare un pezzo difettoso (ad esempio un 'falso negativo', costo stimato ~1000 EUR per un richiamo), molto più grave del segnalare un pezzo conforme come difettoso (un 'falso positivo', costo stimato ~100 EUR per un controllo extra). Questa valutazione, basata sullo specifico pezzo e sul suo utilizzo, porta a preferire un modello che minimizzi i falsi negativi. Per questo ha senso abbassare la soglia di previsione del modello a 0.2 per aumentare significativamente il 'recall' (fino al **69%**), a scapito di precisione e accuratezza. Ciò significa che, a livello teorico e operativo, è meno costoso accettare più falsi positivi per evitare i ben più onerosi falsi negativi (oltre ai rischi per la sicurezza dei clienti).

**Limiti:** Il principale limite riguarda il potere predittivo complessivo non elevatissimo del modello. Il caso di `flatness_mm` e `torque_Nm` descritto sopra, in particolare, evidenzia un limite del metodo di analisi lineare (o, in alternativa, una possibile simulazione del dataset), rendendo difficile cogliere con i soli strumenti statistici univariati tutte le sfumature della difettosità.

### Raccomandazioni operative
1.  **Random Forest con soglia personalizzata:** Suggerirei di implementare il Random Forest con una soglia di decisione calibrata (es. 0.2 o da affinare in base a una valutazione costi/benefici) per massimizzare l'individuazione dei difetti, data la criticità dei falsi negativi.
2.  **Monitoraggio continuo:** È fondamentale monitorare costantemente le prestazioni del modello in produzione, prestando attenzione a come cambiano i dati in ingresso e se il modello degrada le sue prestazioni nel tempo.
3.  **Soglie di intervento articolate:** Propongo di definire **due soglie** di probabilità, creando tre fasce di rischio. I pezzi con probabilità di difetto al di sopra della **soglia superiore** verrebbero **scartati automaticamente**. Quelli con probabilità al di sotto della **soglia inferiore** sarebbero **considerati conformi**. I pezzi con probabilità intermedia verrebbero sottoposti a un **controllo umano approfondito**.
4.  **Retraining periodico:** Il modello dovrebbe essere riaddestrato regolarmente con nuovi dati per mantenere la sua efficacia.
5.  **Privacy dei dati:** È importante garantire la protezione dei dati sensibili, in particolare quelli relativi all'`operator_id`.

### Proposte di ulteriori analisi o dati
1.  **Feature Engineering avanzato:** Esplorare la creazione di nuove variabili più significative dal punto di vista ingegneristico, che possano catturare relazioni complesse.
2.  **Modelli avanzati:** Testare algoritmi più sofisticati come XGBoost o LightGBM, che potrebbero catturare meglio relazioni non lineari.
3.  **Tuning degli iperparametri:** Ottimizzare ulteriormente i parametri dei modelli per massimizzare le prestazioni.
4.  **Raccolta dati aggiuntivi:** Sarebbe utile raccogliere dati sui guasti delle macchine, condizioni ambientali, dati più dettagliati dai sensori e, se possibile, aumentare il numero di campioni di difetti reali.
5.  **Analisi costi-benefici dettagliata:** Collaborare con esperti finanziari e di processo per quantificare con precisione i costi dei falsi positivi e dei falsi negativi, al fine di ottimizzare le soglie del modello.